# SECONDARY → MAIN: Reward Recommendation Model

**Cíl:** Identifikovat SECONDARY zákazníky s nejvyšší šancí stát se MAIN a přiřadit jim optimální odměnu.

**MAIN definice:** příchozí platba ≥ 15 000 Kč + alespoň 3 transakce za měsíc.

**Odměny:**
- `CASH_500` — 500 Kč hotovostní bonus
- `SAVINGS_RATE` — lepší úrok na spoření
- `INVEST_1000` — 1 000 Kč na investice


In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from src.features import build_training_dataset, build_scoring_dataset, MONTHS

DATA_6M     = '../data/VSE_Data_6M.xlsx'
DATA_LABELS = '../data/VSE_Data_LABELY.xlsx'
DATA_DEMO   = '../data/VSE_Data_DEMO.xlsx'

## 1. Exploratory analysis

In [ ]:
df = pd.read_excel(DATA_6M)

seg_counts = {}
for m in MONTHS:
    seg_counts[m] = df[f'PACTSEG_CODE_{m}'].value_counts()

seg_df = pd.DataFrame(seg_counts).T
seg_df.plot(kind='bar', figsize=(10, 4), title='Segment distribution over time')
plt.ylabel('Number of customers')
plt.tight_layout()
plt.show()

In [ ]:
# SECONDARY→MAIN conversion rate per month
conv_rates = {}
for i in range(len(MONTHS) - 1):
    m_from, m_to = MONTHS[i], MONTHS[i+1]
    sec = df[df[f'PACTSEG_CODE_{m_from}'] == 'SECONDARY']
    rate = (sec[f'PACTSEG_CODE_{m_to}'] == 'MAIN').mean()
    conv_rates[f'{m_from}→{m_to}'] = rate

pd.Series(conv_rates).plot(kind='bar', figsize=(8, 3),
    title='Monthly SECONDARY→MAIN conversion rate')
plt.ylabel('Conversion rate')
plt.tight_layout()
plt.show()

In [ ]:
# Feature comparison: converters vs non-converters
X, y_conv, y_reward = build_training_dataset(DATA_6M, DATA_LABELS, DATA_DEMO)

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
features_to_plot = ['avg_cr_turnover', 'avg_spb_login', 'avg_dcrd_usage']
titles = ['Avg CR Turnover (CZK)', 'Avg SPB Logins / month', 'Avg Debit Card Usage']

for ax, feat, title in zip(axes, features_to_plot, titles):
    data = X[[feat]].copy()
    data['converted'] = y_conv.values
    data.boxplot(column=feat, by='converted', ax=ax)
    ax.set_title(title)
    ax.set_xlabel('Converted to MAIN (0/1)')

plt.suptitle('')
plt.tight_layout()
plt.show()

## 2. Train models

In [ ]:
from src.train import train
import os
os.chdir('..')  # run from project root

metrics = train(DATA_6M, DATA_LABELS, DATA_DEMO, cv_folds=5, verbose=True)
print('\nMetrics:', metrics)

In [ ]:
# Feature importance
import joblib
conv_model = joblib.load('../models/conversion_model.joblib')
feature_cols = joblib.load('../models/feature_cols.joblib')

imp = pd.Series(conv_model.feature_importances_, index=feature_cols).sort_values(ascending=True)
imp.tail(15).plot(kind='barh', figsize=(8, 5), title='Feature importance (conversion model)')
plt.tight_layout()
plt.show()

## 3. Score current SECONDARY customers

In [ ]:
from src.predict import score

results = score(DATA_6M, DATA_LABELS, DATA_DEMO, output_csv='../results/scored_customers.csv')
results.head(20)

In [ ]:
print('Tier distribution:')
print(results['conversion_tier'].value_counts())
print()
print('Reward distribution:')
print(results['recommended_reward'].value_counts())
print()
print('Reward by tier:')
print(pd.crosstab(results['conversion_tier'], results['recommended_reward']))

In [ ]:
# Distribution of conversion probabilities
results['conversion_prob'].hist(bins=50, figsize=(8, 3),
    title='Distribution of conversion probabilities')
plt.xlabel('P(SECONDARY → MAIN)')
plt.tight_layout()
plt.show()

## 4. Explain individual customer

In [ ]:
from src.predict import explain_customer
import json

# Pick any SECONDARY customer ID
sample_id = results['ID'].iloc[0]
explanation = explain_customer(sample_id, DATA_6M, DATA_LABELS, DATA_DEMO)
print(json.dumps(explanation, indent=2, ensure_ascii=False, default=str))